# Technique Vocabulary Discovery

**Goal:** Surface candidate technique keywords for each position group that are *not* currently covered by the v1 shared technique keyword list.

**Three discovery methods run in parallel:**

1. **TF-IDF distinctiveness** — terms statistically overrepresented in one position group's scouting language vs. all others
2. **Raw frequency of uncovered tokens** — most common words per position group that fall outside all three existing bins
3. **W2V expansion filtered by position** — technique seed neighbors from the global embedding, ranked by how frequently they appear in each position group's text

**Output:** A single candidate CSV with position group, term, discovery source(s), and frequency — ready to review and paste into the v1 technique keyword list.

In [ ]:
import pandas as pd
import numpy as np
import re
from collections import Counter
from itertools import combinations

from gensim.models import Word2Vec
from sklearn.feature_extraction.text import TfidfVectorizer
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords', quiet=True)
nltk.download('wordnet',   quiet=True)
nltk.download('omw-1.4',   quiet=True)

# ── W2V config (match v1 exactly) ─────────────────────────────────────────────
W2V_DIM       = 100
W2V_WINDOW    = 6
W2V_EPOCHS    = 30
W2V_SG        = 1
W2V_MIN_COUNT = 3
SEED_TOPN     = 30   # slightly wider than v1 for discovery
SIM_THRESHOLD = 0.30 # slightly lower threshold for discovery — cast a wider net

# ── Discovery config ───────────────────────────────────────────────────────────
TFIDF_TOP_N    = 40   # top N TF-IDF distinctive terms to surface per position group
FREQ_TOP_N     = 40   # top N uncovered tokens by raw frequency per position group
W2V_TOP_N      = 20   # top N W2V neighbors per technique seed to inspect
MIN_TERM_FREQ  = 5    # minimum corpus frequency for a candidate to be shown

FILTER_SPECIALISTS = True

print('Imports OK')

## Setup — Preprocessing & Data

Exact same phrase map, stopwords, and preprocess function as `nfl_pillar_Scoring_proportion_v1`. Keep these in sync if v1 changes.

In [ ]:
# ── Curated phrase map (sync with v1) ─────────────────────────────────────────
_CURATED_RAW = {
    'change of direction':'change_of_direction','low pad level':'low_pad_level',
    'run after catch':'run_after_catch','yards after contact':'yards_after_contact',
    'yards after catch':'yards_after_catch','off the line':'off_the_line',
    'off the ball':'off_the_ball','point of attack':'point_of_attack',
    'pass rush':'pass_rush','pass rusher':'pass_rusher',
    'pass protection':'pass_protection','pass coverage':'pass_coverage',
    'pad level':'pad_level','press coverage':'press_coverage',
    'man coverage':'man_coverage','zone coverage':'zone_coverage',
    'ball skills':'ball_skills','ball hawk':'ball_hawk',
    'ball carrier':'ball_carrier','body control':'body_control',
    'contact balance':'contact_balance','closing speed':'closing_speed',
    'lateral quickness':'lateral_quickness','quick twitch':'quick_twitch',
    'high motor':'high_motor','first step':'first_step','get off':'get_off',
    'hand fighting':'hand_fighting','hand strength':'hand_strength',
    'block shedding':'block_shedding','anchor strength':'anchor_strength',
    'route running':'route_running','run blocking':'run_blocking',
    'open field':'open_field','red zone':'red_zone','second level':'second_level',
    'hip flexibility':'hip_flexibility','soft hands':'soft_hands',
    'heavy hands':'heavy_hands','strong hands':'strong_hands',
    'short area':'short_area','three down':'three_down','top end':'top_end',
    'two gap':'two_gap','one gap':'one_gap','snap count':'snap_count',
    'football iq':'football_iq','film study':'film_study',
    'work ethic':'work_ethic','locker room':'locker_room',
    'decision making':'decision_making','play making':'play_making',
    'field vision':'field_vision','play recognition':'play_recognition',
    'pre snap':'pre_snap','post snap':'post_snap',
    'game ready':'game_ready','hard worker':'hard_worker',
}
CURATED_PHRASE_MAP = dict(sorted(_CURATED_RAW.items(), key=lambda x: len(x[0]), reverse=True))

# ── NFL stopwords (sync with v1) ───────────────────────────────────────────────
KEEP_WORDS = {'high','low','heavy','light','deep','short','long','wide','hard',
              'soft','strong','quick','good','great','up','down','off','out',
              'over','through','above','below'}
CUSTOM_STOPS = {'prospect','player','players','show','shows','need','needs',
    'ability','also','often','must','well','still','use','get','make','look',
    'help','time','year','team','game','continue','develop','development',
    'nfl','draft','college','level','type','project','potential','upside','ceiling'}
_base = set(stopwords.words('english'))
NFL_STOPWORDS = (_base - KEEP_WORDS) | CUSTOM_STOPS

lemmatizer = WordNetLemmatizer()

def nfl_preprocess(text: str) -> str:
    if not isinstance(text, str) or not text.strip(): return ''
    text = text.lower()
    text = re.sub(r'[-\u2013\u2014]', ' ', text)
    for phrase, token in CURATED_PHRASE_MAP.items():
        text = text.replace(phrase, token)
    text = re.sub(r'[^a-z_\s]', ' ', text)
    tokens = text.split()
    tokens = [t for t in tokens if '_' in t or t not in NFL_STOPWORDS]
    tokens = [t if '_' in t else lemmatizer.lemmatize(t) for t in tokens]
    tokens = [t for t in tokens if len(t) > 1]
    return ' '.join(tokens)

print('Preprocessing functions ready.')

In [ ]:
df = pd.read_csv('../data/processed/draft_enriched_with_contracts.csv')
keep = ['player_name','Pos_Group','position','grade','year',
        'made_it_contract','overview','strengths','weaknesses']
df = df[keep].copy()
for c in ['overview','strengths','weaknesses']: df[c] = df[c].fillna('')

if FILTER_SPECIALISTS:
    n = len(df)
    df = df[df['Pos_Group'] != 'SPECIAL'].reset_index(drop=True)
    print(f'Removed {n - len(df)} specialists.')

df['all_text']   = (df['overview'].apply(nfl_preprocess) + ' ' +
                    df['strengths'].apply(nfl_preprocess) + ' ' +
                    df['weaknesses'].apply(nfl_preprocess)).str.strip()
df['all_tokens'] = df['all_text'].apply(str.split)

print(f'Players: {len(df)}')
df = df.dropna(subset=['Pos_Group']).reset_index(drop=True)
print(f'Position groups: {sorted(df["Pos_Group"].unique())}')
print(f'Median tokens/player: {df["all_tokens"].apply(len).median():.0f}')

## Reference — Current v1 Keyword Sets

Define all three bin keyword sets from v1 so we can flag candidates already covered and focus only on genuine gaps.

In [ ]:
# ── Current v1 keyword sets — keep in sync with nfl_pillar_Scoring_proportion_v1 ──
# These are used throughout as a coverage filter: any candidate term already
# in these sets is flagged as covered and deprioritised.

CURRENT_PHYSICAL = {
    'explosive','burst','speed','acceleration','first_step','get_off',
    'change_of_direction','agility','frame','size','strength','power',
    'athletic','physical','twitch','height','length','wingspan','weight',
    'build','big','tall','wide','lean','heavy','mass','thick','long',
    'body_control','fluidity','fluid','flexible','flexibility',
    'lateral_quickness','closing_speed','top_end','quick_twitch',
    'natural','raw','powerful','fast','athlete','dynamic','shifty',
    'explosion','vertical','leap','projectable','rare','specimen','elite',
}

CURRENT_TECHNIQUE = {
    'technique','footwork','leverage','pad_level','mechanics',
    'route_running','pass_protection','hand_fighting','block_shedding',
    'anchor_strength','pass_rush','blocking','tackling','coverage',
    'pass_rush','pass_rusher','pass_protection','route_running',
    'run_blocking','block_shedding','anchor_strength','hand_fighting',
    'press_coverage','zone_coverage','man_coverage','low_pad_level',
    'point_of_attack','ball_skills','soft_hands','heavy_hands',
    'stamina','endurance','conditioning','energy','snap_count','three_down',
    'precise','sound','polished','refined','crisp','clean','skilled',
    'technical','fundamental','mechanics','craft','skill','execution','assignment',
}

CURRENT_CHARACTER = {
    'effort','motor','high_motor','relentless','competitive','toughness',
    'instinct','awareness','intelligence','football_iq','coachable',
    'discipline','leadership','work_ethic','recognition',
    'high_motor','work_ethic','hard_worker','hustle','grit','relentless',
    'compete','pursuit','intensity','passion','football_iq','decision_making',
    'play_recognition','pre_snap','post_snap','field_vision','cerebral','smart',
    'savvy','diagnose','read','anticipation','vision','instinct','awareness',
    'recognition','intelligence','leadership','accountable','driven','focused',
    'committed','workhorse','coachable','film_study','locker_room',
    'consistent','reliable','dependable','composure','mature','dedicated',
    'trustworthy','selfless','teammate',
}

ALL_CURRENT = CURRENT_PHYSICAL | CURRENT_TECHNIQUE | CURRENT_CHARACTER

print(f'Current keyword coverage:')
print(f'  physical   : {len(CURRENT_PHYSICAL)} terms')
print(f'  technique  : {len(CURRENT_TECHNIQUE)} terms')
print(f'  character  : {len(CURRENT_CHARACTER)} terms')
print(f'  total unique: {len(ALL_CURRENT)} terms')

In [ ]:
# ── Proper noun blocklist (same logic as v1) ───────────────────────────────────
_cap_count = {}
_total_count = {}
for col in ['overview', 'strengths', 'weaknesses']:
    for text in df[col]:
        if not isinstance(text, str): continue
        for w in text.split():
            w_alpha = re.sub(r'[^a-zA-Z]', '', w)
            if len(w_alpha) < 3: continue
            key = w_alpha.lower()
            _total_count[key] = _total_count.get(key, 0) + 1
            if w_alpha[0].isupper():
                _cap_count[key] = _cap_count.get(key, 0) + 1

PROPER_NOUN_BLOCKLIST = {
    w for w, total in _total_count.items()
    if total >= 5 and _cap_count.get(w, 0) / total >= 0.70
}
print(f'Proper noun blocklist: {len(PROPER_NOUN_BLOCKLIST)} tokens')

## Method 1 — TF-IDF Distinctiveness by Position Group

In [ ]:
# ── TF-IDF: one document per position group ────────────────────────────────────
# Concatenate all preprocessed tokens into a single string per position group.
# TF-IDF then scores each term by how distinctive it is to that group vs. others.

pos_docs = (df.groupby('Pos_Group')['all_text']
              .apply(lambda texts: ' '.join(texts))
              .reset_index())
pos_groups = pos_docs['Pos_Group'].tolist()

vectorizer = TfidfVectorizer(
    token_pattern = r'[a-z_]{2,}',  # allow underscore for stitched phrases
    min_df        = 2,
    max_df        = 0.95,
    sublinear_tf  = True,
)
tfidf_matrix = vectorizer.fit_transform(pos_docs['all_text'])
vocab = vectorizer.get_feature_names_out()

# ── For each position group, extract top-N distinctive terms ──────────────────
tfidf_candidates = []

for i, pg in enumerate(pos_groups):
    row = tfidf_matrix[i].toarray().flatten()
    # sort by TF-IDF score descending
    top_idx = row.argsort()[::-1]
    count = 0
    for idx in top_idx:
        if count >= TFIDF_TOP_N:
            break
        term = vocab[idx]
        score = round(row[idx], 4)
        if score == 0:
            break
        in_physical   = term in CURRENT_PHYSICAL
        in_technique  = term in CURRENT_TECHNIQUE
        in_character  = term in CURRENT_CHARACTER
        tfidf_candidates.append({
            'pos_group'     : pg,
            'term'          : term,
            'tfidf_score'   : score,
            'in_physical'   : in_physical,
            'in_technique'  : in_technique,
            'in_character'  : in_character,
            'already_covered': in_physical or in_technique or in_character,
        })
        count += 1

tfidf_df = pd.DataFrame(tfidf_candidates)

# ── Print uncovered terms only — these are the discovery candidates ────────────
print('Top TF-IDF distinctive terms NOT in any current bin (by position group):')
print('=' * 70)
for pg in sorted(pos_groups):
    subset = (tfidf_df[(tfidf_df['pos_group'] == pg) & (~tfidf_df['already_covered'])]
              .head(20))
    terms = subset['term'].tolist()
    print(f'\n  {pg}: {terms}')

## Method 2 — Uncovered Stitched Phrases by Position Group

Filter to tokens containing an underscore — these are compound phrases that survived preprocessing but aren't in any current bin. Plain unigrams like `run`, `foot`, `hand` are dropped here because they're meaningless without context. Stitched tokens like `two_gapping`, `off_the_ball`, `pass_rushing` are directly actionable.

In [ ]:
# ── Uncovered STITCHED tokens by position group ──────────────────────────────
# Only tokens containing '_' are shown — these are compound phrases that made it
# through preprocessing. Plain unigrams are excluded: they're too ambiguous to be
# useful technique candidates without seeing their full bigram context.

freq_candidates = []

for pg, group_df in df.groupby('Pos_Group'):
    all_tokens_flat = [t for tokens in group_df['all_tokens'] for t in tokens]
    token_counts = Counter(all_tokens_flat)

    for term, count in token_counts.most_common(2000):
        if count < MIN_TERM_FREQ: continue
        if '_' not in term: continue             # stitched phrases only
        if term in ALL_CURRENT: continue         # already covered
        if term in PROPER_NOUN_BLOCKLIST: continue
        base = term.replace('_', '')
        if not base.isalpha(): continue
        freq_candidates.append({
            'pos_group' : pg,
            'term'      : term,
            'freq_count': count,
            'freq_per100': round(count / len(group_df) * 100, 1),
        })

freq_df = pd.DataFrame(freq_candidates)

print('Uncovered stitched phrases by position group (sorted by frequency):')
print('=' * 65)
for pg in sorted(df['Pos_Group'].dropna().unique()):
    subset = freq_df[freq_df['pos_group'] == pg].head(20)
    if subset.empty:
        print(f'\n  {pg}: (none above threshold)')
        continue
    print(f'\n  {pg}:')
    print(f'  {"term":<35} {"count":>6}  {"per 100 players":>15}')
    print(f'  {"-"*58}')
    for _, r in subset.iterrows():
        print(f'  {r["term"]:<35} {r["freq_count"]:>6}  {r["freq_per100"]:>14.1f}%')

## Method 3 — Per-Position PMI Bigram Discovery

Find multi-word expressions that scouts use frequently for each position group but that aren't in the curated phrase map yet. PMI (Pointwise Mutual Information) scores how much more often two words appear together than chance would predict — high PMI = a real collocation, not random co-occurrence.

Run on **raw text before phrase stitching** so we catch bigrams we haven't curated yet. Results are the candidate phrases to consider adding to `CURATED_PHRASE_MAP` in v1.

In [ ]:
from nltk.collocations import BigramCollocationFinder, BigramAssocMeasures

# ── Config ─────────────────────────────────────────────────────────────────────
PMI_MIN_FREQ   = 8    # bigram must appear at least this many times in the position group
PMI_TOP_N      = 30   # top N bigrams to surface per position group

# ── Already-curated phrases — exclude from output ──────────────────────────────
ALREADY_STITCHED = set(k.replace(' ', '_') for k in CURATED_PHRASE_MAP.keys())
ALREADY_STITCHED |= set(CURATED_PHRASE_MAP.values())

# ── Minimal tokenizer for raw text — no stitching, no lemmatization ────────────
# We run PMI on raw tokens to discover NEW bigrams we haven't stitched yet.
def raw_tokenize(text: str) -> list:
    if not isinstance(text, str): return []
    text = text.lower()
    text = re.sub(r'[-\u2013\u2014]', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = text.split()
    # light filter: drop stopwords and very short tokens, keep football words
    tokens = [t for t in tokens
              if len(t) > 2
              and (t not in NFL_STOPWORDS or t in KEEP_WORDS)]
    return tokens

# ── Run per position group ──────────────────────────────────────────────────────
pmi_candidates = []

for pg, group_df in df.groupby('Pos_Group'):
    # Tokenize raw text (all sections combined) for this position group
    token_lists = []
    for _, row in group_df.iterrows():
        combined = (str(row.get('overview','')) + ' ' +
                    str(row.get('strengths','')) + ' ' +
                    str(row.get('weaknesses','')))
        toks = raw_tokenize(combined)
        if toks:
            token_lists.append(toks)

    if len(token_lists) < 10:
        continue

    finder = BigramCollocationFinder.from_documents(token_lists)
    finder.apply_freq_filter(PMI_MIN_FREQ)
    scored = finder.score_ngrams(BigramAssocMeasures.pmi)

    for (w1, w2), score in scored[:PMI_TOP_N]:
        phrase       = f'{w1} {w2}'
        stitched     = f'{w1}_{w2}'
        already_map  = phrase in CURATED_PHRASE_MAP
        already_bin  = stitched in ALL_CURRENT or phrase in ALL_CURRENT
        if w1 in PROPER_NOUN_BLOCKLIST or w2 in PROPER_NOUN_BLOCKLIST:
            continue
        pmi_candidates.append({
            'pos_group'     : pg,
            'bigram'        : phrase,
            'stitched'      : stitched,
            'pmi_score'     : round(score, 3),
            'freq'          : finder.ngram_fd[(w1, w2)],
            'in_phrase_map' : already_map,
            'in_bin'        : already_bin,
            'new'           : not already_map and not already_bin,
        })

pmi_df = pd.DataFrame(pmi_candidates)

# ── Show only NEW bigrams (not already stitched or in any bin) ─────────────────
pmi_new = pmi_df[pmi_df['new']].copy()

print(f'PMI bigrams found: {len(pmi_df)} total, {len(pmi_new)} not in phrase map or bins')
print()
print('NEW technique bigram candidates by position group:')
print('=' * 65)
for pg in sorted(df['Pos_Group'].dropna().unique()):
    subset = pmi_new[pmi_new['pos_group'] == pg].sort_values('pmi_score', ascending=False).head(20)
    if subset.empty:
        continue
    print(f'\n  {pg}:')
    print(f'  {"bigram":<30} {"stitched":<30} {"pmi":>6}  {"freq":>5}')
    print(f'  {"-"*68}')
    for _, r in subset.iterrows():
        print(f'  {r["bigram"]:<30} {r["stitched"]:<30} {r["pmi_score"]:>6.2f}  {r["freq"]:>5}')

## Method 4 — W2V Expansion of Technique Seeds, Filtered by Position Frequency

In [ ]:
# ── Train W2V on full corpus ────────────────────────────────────────────────────
print('Training W2V...')
w2v = Word2Vec(
    sentences   = df['all_tokens'].tolist(),
    vector_size = W2V_DIM,
    window      = W2V_WINDOW,
    epochs      = W2V_EPOCHS,
    sg          = W2V_SG,
    min_count   = W2V_MIN_COUNT,
    workers     = 4,
    seed        = 42,
)
print(f'W2V vocab: {len(w2v.wv)} tokens')

# ── Pre-compute per-position token frequencies ─────────────────────────────────
# For each term returned by W2V, we want to know: how common is it in each
# position group's reports? This lets us rank W2V candidates by position relevance.

pos_token_freq = {}  # {pos_group: Counter(token -> count)}
pos_player_count = {}

for pg, group_df in df.groupby('Pos_Group'):
    all_tokens_flat = [t for tokens in group_df['all_tokens'] for t in tokens]
    pos_token_freq[pg] = Counter(all_tokens_flat)
    pos_player_count[pg] = len(group_df)

print('Position token frequencies computed.')

In [ ]:
# ── Technique anchor seeds (from v1) ───────────────────────────────────────────
TECHNIQUE_SEEDS = [
    'technique','footwork','leverage','pad_level','mechanics',
    'route_running','pass_protection','hand_fighting','block_shedding',
    'anchor_strength','pass_rush','blocking','tackling','coverage',
]

# ── Retrieve W2V neighbors of all technique seeds ──────────────────────────────
w2v_raw_candidates = {}  # {term: [sim scores]}
for seed in TECHNIQUE_SEEDS:
    if seed not in w2v.wv: continue
    for word, sim in w2v.wv.most_similar(seed, topn=SEED_TOPN):
        if sim < SIM_THRESHOLD: continue
        if word in TECHNIQUE_SEEDS: continue
        if word in PROPER_NOUN_BLOCKLIST: continue
        base = word.replace('_', '')
        if not base.isalpha(): continue
        if len(word) <= 2: continue
        w2v_raw_candidates.setdefault(word, []).append(sim)

# ── For each candidate, build a position-frequency profile ─────────────────────
w2v_candidates = []
for term, sims in w2v_raw_candidates.items():
    row = {
        'term'       : term,
        'seed_count' : len(sims),
        'avg_sim'    : round(float(np.mean(sims)), 3),
        'in_technique': term in CURRENT_TECHNIQUE,
        'in_physical' : term in CURRENT_PHYSICAL,
        'in_character': term in CURRENT_CHARACTER,
    }
    # Add per-position frequency
    for pg in sorted(df['Pos_Group'].dropna().unique()):
        cnt   = pos_token_freq[pg].get(term, 0)
        n_pl  = pos_player_count[pg]
        row[f'{pg}_per100'] = round(cnt / n_pl * 100, 2)
    w2v_candidates.append(row)

w2v_df = pd.DataFrame(w2v_candidates).sort_values(['seed_count','avg_sim'], ascending=False)

# Show only terms NOT already in any bin
w2v_new = w2v_df[~(w2v_df['in_technique'] | w2v_df['in_physical'] | w2v_df['in_character'])]

print(f'W2V technique neighbors: {len(w2v_df)} total, {len(w2v_new)} not yet in any bin')
print()
print('New W2V technique candidates (sorted by seed_count, avg_sim):')
per100_cols = [c for c in w2v_new.columns if '_per100' in c]
display_cols = ['term','seed_count','avg_sim'] + per100_cols
print(w2v_new[display_cols].head(40).to_string(index=False))

## Consolidated Candidates

Merge all four discovery methods into one ranked table. Terms appearing in multiple methods (TF-IDF, stitched freq, PMI, W2V) are more reliable candidates — flag them with a `source_count`.

In [ ]:
# ── Build unified candidate table ──────────────────────────────────────────────
all_candidates = {}  # {(pos_group, term): {sources, freq, tfidf, w2v_sim}}

# From TF-IDF
for _, row in tfidf_df[~tfidf_df['already_covered']].iterrows():
    key = (row['pos_group'], row['term'])
    rec = all_candidates.setdefault(key, {'sources': set(), 'tfidf_score': None,
                                          'freq_count': None, 'w2v_avg_sim': None})
    rec['sources'].add('tfidf')
    rec['tfidf_score'] = row['tfidf_score']

# From frequency
for _, row in freq_df.iterrows():
    if row['term'] in ALL_CURRENT: continue
    key = (row['pos_group'], row['term'])
    rec = all_candidates.setdefault(key, {'sources': set(), 'tfidf_score': None,
                                          'freq_count': None, 'w2v_avg_sim': None})
    rec['sources'].add('freq')
    rec['freq_count'] = row['freq_count']


# From PMI bigrams
for _, row in pmi_new.iterrows():
    key = (row['pos_group'], row['stitched'])
    rec = all_candidates.setdefault(key, {'sources': set(), 'tfidf_score': None,
                                          'freq_count': None, 'w2v_avg_sim': None,
                                          'pmi_score': None})
    rec['sources'].add('pmi')
    rec['pmi_score'] = row['pmi_score']

# From W2V (position-agnostic expansion — assign to positions where term is frequent)
for _, row in w2v_new.iterrows():
    per100_cols_here = [c for c in row.index if '_per100' in c]
    for col in per100_cols_here:
        pg = col.replace('_per100', '')
        if row[col] < 1.0: continue  # only assign to positions where it appears in ≥1% of reports
        key = (pg, row['term'])
        rec = all_candidates.setdefault(key, {'sources': set(), 'tfidf_score': None,
                                               'freq_count': None, 'w2v_avg_sim': None})
        rec['sources'].add('w2v')
        rec['w2v_avg_sim'] = row['avg_sim']

# Assemble DataFrame
consolidated_rows = []
for (pg, term), rec in all_candidates.items():
    consolidated_rows.append({
        'pos_group'   : pg,
        'term'        : term,
        'source_count': len(rec['sources']),
        'sources'     : '+'.join(sorted(rec['sources'])),
        'tfidf_score' : rec['tfidf_score'],
        'freq_count'  : rec['freq_count'],
        'w2v_avg_sim' : rec['w2v_avg_sim'],
        'pmi_score'   : rec.get('pmi_score'),
    })

candidates = (pd.DataFrame(consolidated_rows)
              .sort_values(['pos_group','source_count','freq_count'],
                           ascending=[True, False, False])
              .reset_index(drop=True))

# ── Print by position group, highest-confidence first ─────────────────────────
print('CONSOLIDATED TECHNIQUE CANDIDATES')
print('(source_count = how many methods flagged this term — higher = more reliable)')
print('=' * 75)
for pg in sorted(candidates['pos_group'].dropna().unique()):
    sub = candidates[candidates['pos_group'] == pg].head(25)
    print(f'\n  {pg}:')
    print(f'  {"term":<28} {"sources":<18} {"freq":>6}  {"tfidf":>7}  {"w2v_sim":>7}')
    print(f'  {"-"*70}')
    for _, r in sub.iterrows():
        freq = f'{int(r["freq_count"])}' if pd.notna(r['freq_count']) else '—'
        tfidf = f'{r["tfidf_score"]:.4f}' if pd.notna(r['tfidf_score']) else '—'
        w2v   = f'{r["w2v_avg_sim"]:.3f}' if pd.notna(r['w2v_avg_sim']) else '—'
        mark  = '**' if r['source_count'] >= 2 else '  '
        print(f'  {mark}{r["term"]:<26} {r["sources"]:<18} {freq:>6}  {tfidf:>7}  {w2v:>7}')

## Export

Save the consolidated candidate table. Review this CSV, mark which terms belong in the technique bin, then copy them into `MANUAL_ADDITIONS['technique']` in the v1 notebook.

In [ ]:
# ── Save candidates CSV ────────────────────────────────────────────────────────
out_path = '../data/processed/technique_candidates.csv'
candidates.to_csv(out_path, index=False)
print(f'Saved -> {out_path}')
print(f'Shape : {candidates.shape}')
print()

# ── Quick summary ───────────────────────────────────────────────────────────────
print('Candidate counts by position group:')
print(candidates.groupby('pos_group')['term'].count().sort_values(ascending=False).to_string())
print()
print('High-confidence candidates (found by 2+ methods):')
hc = candidates[candidates['source_count'] >= 2]
print(f'  Total: {len(hc)}')
print()
for pg in sorted(hc['pos_group'].dropna().unique()):
    terms = hc[hc['pos_group'] == pg]['term'].tolist()
    print(f'  {pg}: {terms}')